In [14]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import json
import glob
from pprint import pprint

from joblib import Parallel, delayed
from tqdm.auto import tqdm

# from pysolotools.consumers import Solo

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt
# import plotly.express as px
# import plotly.graph_objects as go
# import plotly.io as pio
# pio.renderers.default = 'iframe'

from torchvision.models import resnet50

## Utilites

## Generate Data

In [15]:
SOLO_NAME = 'poisson3_vis'
DATA_PATH = f'../output/SimpleOffice/{SOLO_NAME}'
assert os.path.exists(DATA_PATH), 'data folder not found. $f{DATA_PATH}}'

OUTPUT_PATH = f'./data/SimpleOffice/{SOLO_NAME}'
if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    print(f'create folder: {OUTPUT_PATH}')

In [16]:
from solo_tool import Solo

solo = Solo(DATA_PATH)

/Users/alan/Documents/ELSA/Woven/Scene/WP-16F/py/solo_tool/__init__.py:122: UserWarning: Annotation definition id semantic segmentation not implemented
  warnings.warn(msg)


In [17]:
f = list(solo.frames())[5]
seq_path = f.sequence_path
cap = f.captures[0]
metrics = f.metrics
anno_defs = solo.annotation_definitions
annos = cap.annotations

In [18]:
inst = annos['instance segmentation']
inst.create_masks(seq_path)
# inst.head()
inst_df = inst.instances_df
inst_df.head()

,instanceId,labelId,labelName,color,mask_filename
0,67,14,cup,"[140, 240, 0, 255]",67_cup.png
1,83,4,louver,"[240, 179, 0, 255]",83_louver.png
2,246,2,chair,"[112, 0, 210, 255]",246_chair.png
3,247,23,poster,"[0, 210, 51, 255]",247_poster.png
4,327,2,chair,"[45, 0, 180, 255]",327_chair.png


In [19]:
seg = annos['semantic segmentation']
seg.create_masks(seq_path)
seg_df = seg.instances_df
seg_df.head()

,labelName,pixelValue,mask_filename
0,chair,"[0, 255, 0, 255]",chair.png
1,louver,"[255, 255, 255, 255]",louver.png
2,table,"[226, 255, 73, 255]",table.png
3,cup,"[219, 255, 90, 255]",cup.png
4,paper,"[185, 185, 185, 255]",paper.png


In [20]:
bbox = annos['bounding box']
bbox_df = bbox.values_df
bbox_df.head()

,instanceId,labelId,labelName,x0,y0,w,h,cx,cy
0,67,14,cup,163.0,169.0,12.0,12.0,169.0,175.0
1,83,4,louver,366.0,0.0,74.0,235.0,403.0,117.5
2,246,2,chair,306.0,160.0,72.0,112.0,342.0,216.0
3,247,23,poster,133.0,109.0,61.0,54.0,163.5,136.0
4,327,2,chair,89.0,171.0,53.0,75.0,115.5,208.5


In [21]:
meta = metrics['metadata']
env_meta = meta.env_metadata
meta_df = meta.instances_df
print(env_meta)
meta_df.head()

{'camera': {'focalLength': 31.1548443, 'PanopticVisibleInstanceIds': [206, 205, 204, 83, 527, 207, 326, 325, 344, 20, 211, 348, 335, 371, 247, 369, 537, 368, 476, 13, 242, 38, 41, 236, 372, 338, 64, 237, 347, 60, 111, 62, 127, 39, 130, 245, 289, 1, 129, 235, 329, 265, 171, 43, 34, 336, 97, 286, 40, 315, 30, 246, 209, 29, 428, 170, 173, 466, 427, 17, 465, 327, 215, 463, 533, 374, 67, 340, 342, 431, 445, 8, 474, 444, 532, 169, 334, 526, 44, 45, 42, 390, 447, 504, 439, 529, 58, 330, 424, 534]}}


,instanceId,object_labelName,object_absPos
0,1,storage,"[-15.745, 0.005000591, -12.93]"
1,2,cup,"[-5.389, 0.369441181, 0.411]"
2,3,storage,"[-3.073, 0.005000591, -3.01000023]"
3,4,poster,"[-3.049, 1.81000054, -1.099]"
4,5,storage,"[-2.919, 0.0, -3.789]"


In [22]:
obj_df = meta_df.copy()
# filter invisible objects
visible_inst_ids = inst_df['instanceId']
obj_df = obj_df[obj_df['instanceId'].isin(visible_inst_ids)]

# rename-columns
obj_df = obj_df.rename(columns={'object_labelName': 'label_name'})
obj_df['label_id'] = obj_df['label_name'].apply(lambda x: anno_defs['bounding box'].name2id[x])
pos_df = obj_df['object_absPos'].apply(pd.Series).rename(columns={0: 'pos_x', 1: 'pos_y', 2: 'pos_z'})
obj_df = pd.concat((obj_df, pos_df), axis=1).drop(columns=['object_absPos'])

# merge bbox and instance segmentation
bbox_df_for_merge = bbox_df.copy() \
  .drop(columns=['labelName', 'labelId']) \
  .add_prefix('bbox_') \
  .rename(columns={'bbox_instanceId': 'instanceId'})
inst_df_for_merge = inst_df.copy() \
  .drop(columns=['labelName', 'labelId', 'color']) \
  .add_prefix('inst_') \
  .rename(columns={'inst_instanceId': 'instanceId'})

obj_df = pd.merge(obj_df, bbox_df_for_merge, how='inner', left_on='instanceId', right_on='instanceId', suffixes=('', '_duplicated'))
obj_df = pd.merge(obj_df, inst_df_for_merge, how='inner', left_on='instanceId', right_on='instanceId', suffixes=('', '_duplicated'))

obj_df = obj_df.rename(columns={'instanceId': 'inst_id'})
obj_df

,inst_id,label_name,label_id,pos_x,pos_y,pos_z,bbox_x0,bbox_y0,bbox_w,bbox_h,bbox_cx,bbox_cy,inst_mask_filename
0,67,cup,14,-26.519,0.826707,-10.29100,163.0,169.0,12.0,12.0,169.0,175.0,67_cup.png
1,83,louver,4,-29.110,2.894000,-9.73924,366.0,0.0,74.0,235.0,403.0,117.5,83_louver.png
2,246,chair,2,-28.210,0.005001,-10.42900,306.0,160.0,72.0,112.0,342.0,216.0,246_chair.png
3,247,poster,23,-26.468,1.658001,-12.95100,133.0,109.0,61.0,54.0,163.5,136.0,247_poster.png
4,327,chair,2,-25.790,0.005001,-12.39100,89.0,171.0,53.0,75.0,115.5,208.5,327_chair.png
5,348,poster,23,-27.650,1.923001,-12.95100,201.0,81.0,64.0,83.0,233.0,122.5,348_poster.png
6,428,chair,2,-26.230,0.005001,-11.41300,116.0,165.0,53.0,94.0,142.5,212.0,428_chair.png
7,526,table,7,-27.020,0.005001,-11.37000,135.0,181.0,190.0,108.0,230.0,235.0,526_table.png
8,532,cup,14,-27.571,0.826707,-11.26600,251.0,175.0,8.0,7.0,255.0,178.5,532_cup.png
9,534,paper,20,-28.342,0.005001,-8.88000,440.0,306.0,3.0,2.0,441.5,307.0,534_paper.png


In [23]:
node_df = obj_df.copy()
node_df = pd.concat([pd.Series(node_df.index, name='node_id'), node_df], axis=1)
node2inst = node_df[['node_id', 'inst_id']].to_dict()['inst_id']

node_df.head()

,node_id,inst_id,label_name,label_id,pos_x,pos_y,pos_z,bbox_x0,bbox_y0,bbox_w,bbox_h,bbox_cx,bbox_cy,inst_mask_filename
0,0,67,cup,14,-26.519,0.826707,-10.29100,163.0,169.0,12.0,12.0,169.0,175.0,67_cup.png
1,1,83,louver,4,-29.110,2.894000,-9.73924,366.0,0.0,74.0,235.0,403.0,117.5,83_louver.png
2,2,246,chair,2,-28.210,0.005001,-10.42900,306.0,160.0,72.0,112.0,342.0,216.0,246_chair.png
3,3,247,poster,23,-26.468,1.658001,-12.95100,133.0,109.0,61.0,54.0,163.5,136.0,247_poster.png
4,4,327,chair,2,-25.790,0.005001,-12.39100,89.0,171.0,53.0,75.0,115.5,208.5,327_chair.png


In [24]:
diff = cross_df[['bbox_cx_dst', 'bbox_cy_dst']].values - cross_df[['bbox_cx_src', 'bbox_cy_src']].values
r = np.linalg.norm(diff, axis=1)
theta = np.arctan2(diff[:, 1], diff[:, 0])
sin, cos = np.sin(theta), np.cos(theta)
df = pd.DataFrame({'dist':r, 'sin':sin, 'cos':cos}, index=cross_df.index)
df = pd.concat((cross_df[['node_id_src', 'node_id_dst']], df), axis=1)
edge_df.merge(df, on=['node_id_src', 'node_id_dst'], how='inner')

NameError: name 'cross_df' is not defined

In [ ]:
# construct edge pair id
node_df_for_cross = node_df.copy()[['node_id', 'bbox_cx', 'bbox_cy']]
cross_df = node_df_for_cross.merge(node_df_for_cross, how='cross', suffixes=('_src', '_dst'))
# construct edge features
diff = cross_df[['bbox_cx_dst', 'bbox_cy_dst']].values - cross_df[['bbox_cx_src', 'bbox_cy_src']].values
cross_df['bbox_dist'] = np.linalg.norm(diff, axis=1)
theta = np.arctan2(diff[:, 1], diff[:, 0])
cross_df['bbox_sin'] = np.sin(theta)
cross_df['bbox_cos'] = np.cos(theta)

# bbox dist as edge weight
dist_pivot = cross_df.pivot(index='node_id_dst', columns='node_id_src', values='bbox_dist')
# find first 3 pair with minimum weight
knn_bbox_dist = dist_pivot.apply(lambda x: x.nsmallest(1+3).index).T
edge_df = knn_bbox_dist.melt(id_vars=[0], value_vars=[1, 2])[[0, 'value']].rename(columns={0: 'node_id_src', 'value': 'node_id_dst'})

# drop duplicated bi-directional edge
# edge['small2large_id'] = edge.apply(lambda x: (x['node_id_src'], x['node_id_dst']) if (x['node_id_src'] < x['node_id_dst']) else (x['node_id_dst'], x['node_id_src']), axis=1)
# edge.drop_duplicates(subset=['small2large_id'], inplace=True)

edge_df.merge(cross_df[['node_id_src', 'node_id_dst', 'bbox_dist', 'bbox_sin', 'bbox_cos']], on=['node_id_src', 'node_id_dst'], how='inner')
# edge_df

In [1]:
import copy
from solo_tool import Solo
from solo2graph import frame_to_data, pair_graph

/Users/alan/Documents/ELSA/Woven/Scene/WP-16F/py/venv/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
SOLO_NAME = 'poisson3r8_vis'
DATA_PATH = f'../output/SimpleOffice/{SOLO_NAME}'
solo = Solo(DATA_PATH)

graphs = []
for f in solo.frames():
  valid, data_dict = frame_to_data(f, solo)
  break
  # if not valid:
  #   print(f'skip {f.frame}')
  #   continue
  # graphs.append(data_dict)
# data_dict['node_df']

/Users/alan/Documents/ELSA/Woven/Scene/WP-16F/py/solo_tool/__init__.py:122: UserWarning: Annotation definition id semantic segmentation not implemented
  warnings.warn(msg)
/Users/alan/Documents/ELSA/Woven/Scene/WP-16F/py/venv/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


torch.Size([3, 337, 439]) torch.float32
(34, 1000)


,node_id,inst_id,label_name,label_id,pos_x,pos_y,pos_z,bbox_x0,bbox_y0,bbox_w,bbox_h,bbox_cx,bbox_cy,inst_mask_filename
0,0,494,box,11,-25.183000,1.657001,-13.752000,386.0,128.0,44.0,24.0,408.0,140.0,494_box.png
1,1,495,box,11,-25.244001,1.164001,-14.318000,412.0,194.0,27.0,21.0,425.5,204.5,495_box.png
2,2,496,box,11,-25.244001,1.290001,-14.318000,414.0,179.0,25.0,18.0,426.5,188.0,496_box.png
3,3,497,box,11,-25.206001,1.164001,-13.267000,359.0,179.0,38.0,20.0,378.0,189.0,497_box.png
4,4,502,box,11,-25.202002,0.179001,-13.260000,359.0,263.0,35.0,25.0,376.5,275.5,502_box.png
5,5,503,box,11,-25.202002,0.179001,-13.491000,369.0,270.0,6.0,23.0,372.0,281.5,503_box.png
6,6,504,box,11,-25.202002,0.179001,-13.798000,375.0,268.0,64.0,47.0,407.0,291.5,504_box.png
7,7,505,box,11,-25.200000,0.670001,-13.822000,386.0,235.0,51.0,28.0,411.5,249.0,505_box.png
8,8,506,box,11,-24.690000,0.670001,-13.361000,405.0,229.0,34.0,19.0,422.0,238.5,506_box.png
9,9,507,box,11,-24.696701,1.164401,-13.790800,431.0,176.0,8.0,3.0,435.0,177.5,507_box.png


In [ ]:
g1 = graphs[28]
g2 = graphs[29]
overlap, dc = pair_graph(g1, g2)
dc